# sampling rate 로 데이터 sampling 하는 코드

In [ ]:
from datasets import load_from_disk
import os
import random
import multiprocessing

# 샘플링 비율 (1/100 = 1%)
SAMPLE_RATIO = 0.1
SEED = 42

# 모든 split에서 task별 최소 샘플 개수 (0개 방지)
MIN_SAMPLES_PER_TASK = 16

# 병렬 처리 설정
NUM_PROC = min(multiprocessing.cpu_count(), 50)  # CPU 코어 수 (최대 16)
BATCH_SIZE = 10000  # 배치 크기

# 경로 설정
BASE_DIR = "/home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official"
OUTPUT_DIR = "/home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official"

# split 종류
SPLITS = ["train", "test", "val"]

# 파일명 패턴 (total_merged 데이터셋 사용)
PREFIX = "GSAI-ML-LLaDA-8B-Instruct_string+graph_q32"
SUFFIX = "512_Truncation"

# 출력 파일명 suffix
OUTPUT_SUFFIX = "10percent_sampled"


def get_selected_indices(ds, min_samples: int = 3, sample_ratio: float = 0.01):
    """각 task별로 최소 min_samples개를 보장하면서 샘플링할 인덱스 반환"""
    random.seed(SEED)

    # task별로 인덱스 수집
    task_to_indices = {}
    for idx, task in enumerate(ds['task']):
        if task not in task_to_indices:
            task_to_indices[task] = []
        task_to_indices[task].append(idx)

    # 각 task에서 샘플 선택
    selected_indices = set()
    task_selection_info = {}

    for task, indices in task_to_indices.items():
        ratio_samples = int(len(indices) * sample_ratio)
        n_select = min(len(indices), max(min_samples, ratio_samples))
        n_select = max(1, n_select)

        random.shuffle(indices)
        selected_indices.update(indices[:n_select])
        task_selection_info[task] = (len(indices), n_select)

    return selected_indices, task_selection_info


def sample_dataset_for_split(split: str):
    """특정 split에 대해 전체 데이터의 1% 샘플링 (filter 사용)"""

    input_name = f"{PREFIX}_{split}_{SUFFIX}"
    input_path = os.path.join(BASE_DIR, input_name)

    if not os.path.exists(input_path):
        print(f"  [에러] 경로 없음: {input_path}")
        return None

    print(f"  로딩: {input_path}")
    ds = load_from_disk(input_path)
    original_size = len(ds)
    print(f"    - 전체 샘플 수: {original_size}")

    print(f"  Task별 최소 {MIN_SAMPLES_PER_TASK}개 샘플 보장 샘플링...")
    selected_indices, task_info = get_selected_indices(
        ds, min_samples=MIN_SAMPLES_PER_TASK, sample_ratio=SAMPLE_RATIO
    )

    print(f"    - 샘플링할 샘플 수: {len(selected_indices)}")

    # filter를 사용하여 샘플링 (메타데이터 유지, 병렬 처리)
    print(f"  필터링 중... (num_proc={NUM_PROC}, batch_size={BATCH_SIZE})")
    sampled_ds = ds.filter(
        lambda example, indices: [idx in selected_indices for idx in indices],
        with_indices=True,
        num_proc=NUM_PROC,
        batched=True,
        batch_size=BATCH_SIZE
    )

    print(f"    - 샘플링 후 샘플 수: {len(sampled_ds)}")

    # task별 개수 출력
    print(f"    - Task별 개수 (원본 -> 샘플링):")
    task_counts = {}
    for task in sampled_ds['task']:
        task_counts[task] = task_counts.get(task, 0) + 1

    for task, count in sorted(task_counts.items()):
        orig_count = task_info.get(task, (0, 0))[0]
        print(f"      {task}: {orig_count} -> {count}")

    # 누락된 task 확인
    missing_tasks = set(task_info.keys()) - set(task_counts.keys())
    if missing_tasks:
        print(f"  [경고] 누락된 task: {missing_tasks}")
    else:
        print(f"  [OK] 모든 task가 포함됨!")

    return sampled_ds


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 60)
    print("전체 데이터셋 1% 샘플링 시작")
    print(f"샘플링 비율: {SAMPLE_RATIO*100:.1f}%")
    print(f"Task별 최소 샘플 수: {MIN_SAMPLES_PER_TASK}")
    print(f"병렬 처리: num_proc={NUM_PROC}, batch_size={BATCH_SIZE}")
    print("=" * 60)

    for split in SPLITS:
        print(f"\n[{split.upper()}] 샘플링 중...")

        sampled_dataset = sample_dataset_for_split(split)

        if sampled_dataset is not None:
            output_name = f"{PREFIX}_{split}_{SUFFIX}_{OUTPUT_SUFFIX}"
            output_path = os.path.join(OUTPUT_DIR, output_name)

            if os.path.exists(output_path):
                print(f"  기존 데이터셋 삭제 중...")
                import shutil
                shutil.rmtree(output_path)

            # 저장 (병렬 처리)
            print(f"  저장 중: {output_path} (num_proc={NUM_PROC})")
            sampled_dataset.save_to_disk(output_path, num_proc=NUM_PROC)
            print(f"  저장 완료!")

    print("\n" + "=" * 60)
    print("모든 샘플링 완료!")
    print("=" * 60)

    # 결과 확인
    print("\n[결과 확인]")
    for split in SPLITS:
        output_name = f"{PREFIX}_{split}_{SUFFIX}_{OUTPUT_SUFFIX}"
        output_path = os.path.join(OUTPUT_DIR, output_name)
        if os.path.exists(output_path):
            try:
                ds = load_from_disk(output_path)
                print(f"\n  === {split.upper()} ({len(ds)} 샘플) ===")

                task_counts = {}
                for task in ds['task']:
                    task_counts[task] = task_counts.get(task, 0) + 1

                for task, count in sorted(task_counts.items()):
                    print(f"    {task}: {count}")
            except Exception as e:
                print(f"\n  === {split.upper()} 로드 실패: {e} ===")


if __name__ == "__main__":
    main()


# 단일 Task만 filtering해서 데이터 만드는 코드


In [4]:
from datasets import load_from_disk
import os

# Validation set 샘플링 비율
VAL_SAMPLE_RATIO = 0.1
SEED = 42

# 경로 설정
BASE_DIR = "/home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official"
OUTPUT_DIR = "/home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official"

# 필터링할 task 목록
TARGET_TASKS = ["chebi-20-mol2text"]

# split 종류
SPLITS = ["train", "test", "val"]

# 파일명 패턴
PREFIX = "GSAI-ML-LLaDA-8B-Instruct_string+graph_q32"
SUFFIX = "512_Truncation"

# 출력 파일명 suffix
OUTPUT_SUFFIX = "chebi_mol2text_10pct"


def filter_dataset_for_split(split: str):
    """특정 split에 대해 chebi task만 필터링"""

    # 입력 경로
    input_name = f"{PREFIX}_{split}_{SUFFIX}"
    input_path = os.path.join(BASE_DIR, input_name)

    if not os.path.exists(input_path):
        print(f"  [에러] 경로 없음: {input_path}")
        return None

    print(f"  로딩: {input_path}")
    ds = load_from_disk(input_path)
    print(f"    - 전체 샘플 수: {len(ds)}")

    # Dataset.filter()를 사용하여 원본 features 스키마 유지 (batched + multiprocessing)
    print(f"  필터링 중... (tasks: {TARGET_TASKS})")
    target_set = set(TARGET_TASKS)
    filtered_ds = ds.filter(
        lambda batch: [t in target_set for t in batch['task']],
        batched=True,
        batch_size=10000,
        num_proc=50
    )
    print(f"    - 필터링 후 샘플 수: {len(filtered_ds)}")

    if len(filtered_ds) == 0:
        print(f"  [경고] 필터링 결과가 비어있습니다.")
        return None

    # task별 개수 출력
    print(f"    - Task별 개수:")
    task_counts = {}
    for task in filtered_ds['task']:
        task_counts[task] = task_counts.get(task, 0) + 1
    for task in TARGET_TASKS:
        print(f"      {task}: {task_counts.get(task, 0)}")

    return filtered_ds


def main():
    # 출력 디렉토리 생성
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 60)
    print("Chebi 데이터셋 필터링 시작")
    print(f"대상 Task: {TARGET_TASKS}")
    print("=" * 60)

    for split in SPLITS:
        print(f"\n[{split.upper()}] 필터링 중...")

        filtered_dataset = filter_dataset_for_split(split)

        if filtered_dataset is not None:
            # Validation set은 10%만 랜덤 샘플링
            if split == "val":
                original_size = len(filtered_dataset)
                sample_size = int(original_size * VAL_SAMPLE_RATIO)
                print(f"  Validation set 샘플링: {original_size} -> {sample_size} ({VAL_SAMPLE_RATIO*100:.0f}%)")
                filtered_dataset = filtered_dataset.shuffle(seed=SEED).select(range(sample_size))

                # 샘플링 후 task별 개수 재출력
                print(f"    - 샘플링 후 Task별 개수:")
                task_counts = {}
                for task in filtered_dataset['task']:
                    task_counts[task] = task_counts.get(task, 0) + 1
                for task in TARGET_TASKS:
                    print(f"      {task}: {task_counts.get(task, 0)}")

            # 출력 경로 설정
            output_name = f"{PREFIX}_{split}_{SUFFIX}_{OUTPUT_SUFFIX}"
            output_path = os.path.join(OUTPUT_DIR, output_name)

            # 저장
            print(f"  저장 중: {output_path}")
            filtered_dataset.save_to_disk(output_path)
            print(f"  저장 완료!")

    print("\n" + "=" * 60)
    print("모든 필터링 완료!")
    print("=" * 60)

    # 결과 확인
    print("\n[결과 확인]")
    for split in SPLITS:
        output_name = f"{PREFIX}_{split}_{SUFFIX}_{OUTPUT_SUFFIX}"
        output_path = os.path.join(OUTPUT_DIR, output_name)
        if os.path.exists(output_path):
            ds = load_from_disk(output_path)
            print(f"  {split}: {len(ds)} 샘플")


if __name__ == "__main__":
    main()


Chebi 데이터셋 필터링 시작
대상 Task: ['chebi-20-mol2text']

[TRAIN] 필터링 중...
  로딩: /home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official/GSAI-ML-LLaDA-8B-Instruct_string+graph_q32_train_512_Truncation


Loading dataset from disk:   0%|          | 0/43 [00:00<?, ?it/s]

    - 전체 샘플 수: 3303537
  필터링 중... (tasks: ['chebi-20-mol2text'])
    - 필터링 후 샘플 수: 26113
    - Task별 개수:
      chebi-20-mol2text: 26113
  저장 중: /home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official/GSAI-ML-LLaDA-8B-Instruct_string+graph_q32_train_512_Truncation_chebi_mol2text_10pct


Saving the dataset (0/1 shards):   0%|          | 0/26113 [00:00<?, ? examples/s]

  저장 완료!

[TEST] 필터링 중...
  로딩: /home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official/GSAI-ML-LLaDA-8B-Instruct_string+graph_q32_test_512_Truncation
    - 전체 샘플 수: 32595
  필터링 중... (tasks: ['chebi-20-mol2text'])
    - 필터링 후 샘플 수: 3273
    - Task별 개수:
      chebi-20-mol2text: 3273
  저장 중: /home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official/GSAI-ML-LLaDA-8B-Instruct_string+graph_q32_test_512_Truncation_chebi_mol2text_10pct


Saving the dataset (0/1 shards):   0%|          | 0/3273 [00:00<?, ? examples/s]

  저장 완료!

[VAL] 필터링 중...
  로딩: /home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official/GSAI-ML-LLaDA-8B-Instruct_string+graph_q32_val_512_Truncation
    - 전체 샘플 수: 35042
  필터링 중... (tasks: ['chebi-20-mol2text'])
    - 필터링 후 샘플 수: 3003
    - Task별 개수:
      chebi-20-mol2text: 3003
  Validation set 샘플링: 3003 -> 300 (10%)
    - 샘플링 후 Task별 개수:
      chebi-20-mol2text: 300
  저장 중: /home/jovyan/CHJ/Mol-LLM_Custom/dataset/train_official/GSAI-ML-LLaDA-8B-Instruct_string+graph_q32_val_512_Truncation_chebi_mol2text_10pct


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

  저장 완료!

모든 필터링 완료!

[결과 확인]
  train: 26113 샘플
  test: 3273 샘플
  val: 300 샘플
